# Preprocesamiento: Restaurant Tips Dataset

## 🎯 Objetivos de Aprendizaje
- **Técnica Principal**: Análisis de Relaciones entre Variables + Predicción de Propinas
- **Dataset**: Restaurant Tips (Tips dataset)
- **Nivel**: Intermedio

## 📊 Contexto del Dataset
El dataset de propinas contiene información sobre facturas y propinas en restaurantes. Es ideal para aprender:
- **Análisis exploratorio** con variables mixtas (numéricas y categóricas)
- **Visualizaciones avanzadas** para entender relaciones
- **Preprocesamiento** de variables categóricas y numéricas
- **Modelos de regresión** y clasificación
- **Feature engineering** (crear nuevas variables)

## 💡 ¿Por qué Tips Dataset?
- ✅ Variables mixtas reales (numéricas + categóricas)
- ✅ Contexto familiar y fácil de interpretar
- ✅ Problemas típicos del mundo real (outliers, distribuciones sesgadas)
- ✅ Múltiples objetivos posibles (regresión y clasificación)
- ✅ Oportunidades para feature engineering


In [ ]:
# 📦 Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

## 📊 Paso 1: Carga y Exploración Inicial

**🎯 Enfoque**: Comprensión inicial del dataset y identificación de oportunidades de preprocesamiento.

In [ ]:
# Cargar el dataset
print("🍽️ Cargando Restaurant Tips Dataset...")

# El dataset está disponible en seaborn
tips = sns.load_dataset('tips')

print(f"✅ Dataset cargado exitosamente")
print(f"📊 Shape del dataset: {tips.shape}")
print(f"📋 Columnas: {list(tips.columns)}")

# Información básica
print(f"\n📈 Información del Dataset:")
print(f"  - Total de registros: {len(tips)}")
print(f"  - Total de características: {len(tips.columns)}")
print(f"  - Tipos de datos únicos: {tips.dtypes.nunique()}")

# Verificar valores faltantes
missing_values = tips.isnull().sum()
print(f"\n🔍 Análisis de valores faltantes:")
if missing_values.sum() == 0:
    print("  ✅ Dataset completo - sin valores faltantes")
else:
    print("  ⚠️ Valores faltantes encontrados:")
    for col, missing in missing_values[missing_values > 0].items():
        percentage = (missing / len(tips)) * 100
        print(f"    - {col}: {missing} ({percentage:.1f}%)")

# Mostrar las primeras filas
print(f"\n📋 Primeras 5 filas del dataset:")
print(tips.head())

# Información de tipos de datos
print(f"\n📊 Tipos de datos:")
for dtype, count in tips.dtypes.value_counts().items():
    print(f"  - {dtype}: {count} columnas")

# Información detallada del dataset
print(f"\n📋 Información detallada:")
tips.info()

## 📊 Paso 2: Análisis Exploratorio de Datos (EDA)

**🎯 Enfoque**: Análisis profundo de las distribuciones, relaciones y patrones en los datos.

In [ ]:
# Estadísticas descriptivas
print("📊 ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 40)

# Variables numéricas
numeric_cols = tips.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n🔢 Variables numéricas: {numeric_cols}")
print(tips[numeric_cols].describe().round(2))

# Variables categóricas
categorical_cols = tips.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\n🏷️ Variables categóricas: {categorical_cols}")

for col in categorical_cols:
    print(f"\n📋 {col}:")
    value_counts = tips[col].value_counts()
    for value, count in value_counts.items():
        percentage = (count / len(tips)) * 100
        print(f"  - {value}: {count} ({percentage:.1f}%)")

# Análisis de variables target potenciales
print(f"\n🎯 ANÁLISIS DE VARIABLES TARGET:")
print(f"\n💰 Tip (monto):")
print(f"  - Rango: ${tips['tip'].min():.2f} - ${tips['tip'].max():.2f}")
print(f"  - Promedio: ${tips['tip'].mean():.2f}")
print(f"  - Mediana: ${tips['tip'].median():.2f}")
print(f"  - Desviación estándar: ${tips['tip'].std():.2f}")

# Calcular tip percentage
tips['tip_percentage'] = (tips['tip'] / tips['total_bill']) * 100
print(f"\n📊 Tip Percentage (%):")
print(f"  - Rango: {tips['tip_percentage'].min():.1f}% - {tips['tip_percentage'].max():.1f}%")
print(f"  - Promedio: {tips['tip_percentage'].mean():.1f}%")
print(f"  - Mediana: {tips['tip_percentage'].median():.1f}%")
print(f"  - Desviación estándar: {tips['tip_percentage'].std():.1f}%")

In [ ]:
# Visualizaciones exploratorias
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Análisis Exploratorio - Restaurant Tips Dataset', fontsize=16, y=0.98)

# 1. Distribución del monto de la cuenta
axes[0,0].hist(tips['total_bill'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
axes[0,0].set_title('Distribución del Total de la Cuenta')
axes[0,0].set_xlabel('Total Bill ($)')
axes[0,0].set_ylabel('Frecuencia')
axes[0,0].axvline(tips['total_bill'].mean(), color='red', linestyle='--', 
                 label=f'Media: ${tips["total_bill"].mean():.2f}')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Distribución de la propina
axes[0,1].hist(tips['tip'], bins=20, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0,1].set_title('Distribución de la Propina')
axes[0,1].set_xlabel('Tip ($)')
axes[0,1].set_ylabel('Frecuencia')
axes[0,1].axvline(tips['tip'].mean(), color='red', linestyle='--', 
                label=f'Media: ${tips["tip"].mean():.2f}')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Distribución del porcentaje de propina
axes[0,2].hist(tips['tip_percentage'], bins=20, alpha=0.7, color='orange', edgecolor='black')
axes[0,2].set_title('Distribución del Porcentaje de Propina')
axes[0,2].set_xlabel('Tip Percentage (%)')
axes[0,2].set_ylabel('Frecuencia')
axes[0,2].axvline(tips['tip_percentage'].mean(), color='red', linestyle='--', 
                label=f'Media: {tips["tip_percentage"].mean():.1f}%')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# 4. Propinas por día de la semana
day_order = ['Thur', 'Fri', 'Sat', 'Sun']
tips_by_day = tips.groupby('day')['tip'].mean().reindex(day_order)
axes[1,0].bar(tips_by_day.index, tips_by_day.values, color='purple', alpha=0.7)
axes[1,0].set_title('Propina Promedio por Día')
axes[1,0].set_xlabel('Día')
axes[1,0].set_ylabel('Tip Promedio ($)')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].grid(True, alpha=0.3)

# Añadir valores en las barras
for i, v in enumerate(tips_by_day.values):
    axes[1,0].text(i, v + 0.05, f'${v:.2f}', ha='center', va='bottom')

# 5. Propinas por género del cliente
tips_by_sex = tips.groupby('sex')['tip'].mean()
axes[1,1].bar(tips_by_sex.index, tips_by_sex.values, color='teal', alpha=0.7)
axes[1,1].set_title('Propina Promedio por Género')
axes[1,1].set_xlabel('Género')
axes[1,1].set_ylabel('Tip Promedio ($)')
axes[1,1].grid(True, alpha=0.3)

# Añadir valores en las barras
for i, v in enumerate(tips_by_sex.values):
    axes[1,1].text(i, v + 0.05, f'${v:.2f}', ha='center', va='bottom')

# 6. Propinas por si es fumador
tips_by_smoker = tips.groupby('smoker')['tip'].mean()
axes[1,2].bar(tips_by_smoker.index, tips_by_smoker.values, color='coral', alpha=0.7)
axes[1,2].set_title('Propina Promedio por Hábito de Fumar')
axes[1,2].set_xlabel('Fumador')
axes[1,2].set_ylabel('Tip Promedio ($)')
axes[1,2].grid(True, alpha=0.3)

# Añadir valores en las barras
for i, v in enumerate(tips_by_smoker.values):
    axes[1,2].text(i, v + 0.05, f'${v:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 🔍 Paso 3: Análisis de Relaciones entre Variables

**🎯 Enfoque**: Identificación de patrones y correlaciones que guíen el feature engineering.

In [ ]:
# Análisis de correlaciones
print("🔗 ANÁLISIS DE CORRELACIONES")
print("=" * 40)

# Matriz de correlación
numeric_data = tips[['total_bill', 'tip', 'tip_percentage', 'size']]
correlation_matrix = numeric_data.corr()

print(f"\n📊 Matriz de correlación:")
print(correlation_matrix.round(3))

# Visualización de correlaciones
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8}, fmt='.3f')
plt.title('Matriz de Correlación - Variables Numéricas')
plt.tight_layout()
plt.show()

# Análisis de correlación con la variable target (tip)
print(f"\n🎯 Correlaciones con 'tip' (monto de propina):")
tip_correlations = correlation_matrix['tip'].sort_values(ascending=False)
for var, corr in tip_correlations.items():
    if var != 'tip':
        print(f"  - {var}: {corr:.3f}")

print(f"\n🎯 Correlaciones con 'tip_percentage':")
tip_pct_correlations = correlation_matrix['tip_percentage'].sort_values(ascending=False)
for var, corr in tip_pct_correlations.items():
    if var != 'tip_percentage':
        print(f"  - {var}: {corr:.3f}")

In [ ]:
# Análisis de relaciones categóricas con variables numéricas
print(f"📊 ANÁLISIS DE VARIABLES CATEGÓRICAS")
print("=" * 45)

# Box plots para variables categóricas
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análisis de Variables Categóricas vs Propina', fontsize=16)

# 1. Propina por día de la semana
sns.boxplot(data=tips, x='day', y='tip', ax=axes[0,0], order=day_order)
axes[0,0].set_title('Distribución de Propinas por Día')
axes[0,0].set_xlabel('Día')
axes[0,0].set_ylabel('Tip ($)')

# 2. Propina por tiempo
sns.boxplot(data=tips, x='time', y='tip', ax=axes[0,1])
axes[0,1].set_title('Distribución de Propinas por Tiempo')
axes[0,1].set_xlabel('Tiempo')
axes[0,1].set_ylabel('Tip ($)')

# 3. Propina por género
sns.boxplot(data=tips, x='sex', y='tip', ax=axes[1,0])
axes[1,0].set_title('Distribución de Propinas por Género')
axes[1,0].set_xlabel('Género')
axes[1,0].set_ylabel('Tip ($)')

# 4. Propina por hábito de fumar
sns.boxplot(data=tips, x='smoker', y='tip', ax=axes[1,1])
axes[1,1].set_title('Distribución de Propinas por Hábito de Fumar')
axes[1,1].set_xlabel('Fumador')
axes[1,1].set_ylabel('Tip ($)')

plt.tight_layout()
plt.show()

# Análisis estadístico de diferencias entre grupos
print(f"\n📈 ANÁLISIS ESTADÍSTICO POR GRUPO")
print("=" * 45)

# Test de diferencia de medias para género
from scipy.stats import ttest_ind

male_tips = tips[tips['sex'] == 'Male']['tip']
female_tips = tips[tips['sex'] == 'Female']['tip']

t_stat, p_value = ttest_ind(male_tips, female_tips)
print(f"\n🎯 Diferencia de propinas por género:")
print(f"  - Male: μ=${male_tips.mean():.2f}, σ=${male_tips.std():.2f}")
print(f"  - Female: μ=${female_tips.mean():.2f}, σ=${female_tips.std():.2f}")
print(f"  - Test t: estadístico={t_stat:.3f}, p-valor={p_value:.3f}")
print(f"  - Diferencia significativa: {'Sí' if p_value < 0.05 else 'No'}")

# Análisis por tiempo
lunch_tips = tips[tips['time'] == 'Lunch']['tip']
dinner_tips = tips[tips['time'] == 'Dinner']['tip']

t_stat2, p_value2 = ttest_ind(lunch_tips, dinner_tips)
print(f"\n🎯 Diferencia de propinas por tiempo:")
print(f"  - Lunch: μ=${lunch_tips.mean():.2f}, σ=${lunch_tips.std():.2f}")
print(f"  - Dinner: μ=${dinner_tips.mean():.2f}, σ=${dinner_tips.std():.2f}")
print(f"  - Test t: estadístico={t_stat2:.3f}, p-valor={p_value2:.3f}")
print(f"  - Diferencia significativa: {'Sí' if p_value2 < 0.05 else 'No'}")

In [ ]:
# Scatter plots para relaciones numéricas
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Relaciones entre Variables Numéricas', fontsize=16)

# 1. Total Bill vs Tip
axes[0].scatter(tips['total_bill'], tips['tip'], alpha=0.6, color='blue')
axes[0].set_xlabel('Total Bill ($)')
axes[0].set_ylabel('Tip ($)')
axes[0].set_title('Total Bill vs Tip')
axes[0].grid(True, alpha=0.3)

# Añadir línea de regresión
z = np.polyfit(tips['total_bill'], tips['tip'], 1)
p = np.poly1d(z)
axes[0].plot(tips['total_bill'], p(tips['total_bill']), "r--", alpha=0.8)

# 2. Size vs Tip
axes[1].scatter(tips['size'], tips['tip'], alpha=0.6, color='green')
axes[1].set_xlabel('Size (Número de personas)')
axes[1].set_ylabel('Tip ($)')
axes[1].set_title('Size vs Tip')
axes[1].grid(True, alpha=0.3)

# 3. Total Bill vs Tip Percentage
axes[2].scatter(tips['total_bill'], tips['tip_percentage'], alpha=0.6, color='orange')
axes[2].set_xlabel('Total Bill ($)')
axes[2].set_ylabel('Tip Percentage (%)')
axes[2].set_title('Total Bill vs Tip Percentage')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Análisis de outliers
print(f"\n🔍 ANÁLISIS DE OUTLIERS")
print("=" * 30)

def detect_outliers_iqr(data, column):
    """Detecta outliers usando el método IQR"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, len(outliers)

for col in ['total_bill', 'tip', 'tip_percentage']:
    outliers, count = detect_outliers_iqr(tips, col)
    percentage = (count / len(tips)) * 100
    print(f"\n{col}:")
    print(f"  - Outliers detectados: {count} ({percentage:.1f}%)")
    if count > 0:
        print(f"  - Valores outliers: {outliers[col].values[:5]}{'...' if count > 5 else ''}")

## 🔧 Paso 4: Feature Engineering

**🎯 Enfoque**: Creación de nuevas variables que mejoren el poder predictivo del modelo.

In [ ]:
# Feature Engineering
print(f"🛠️ FEATURE ENGINEERING")
print("=" * 35)

tips_enhanced = tips.copy()

# 1. Categorizar el monto de la cuenta
tips_enhanced['bill_category'] = pd.cut(tips_enhanced['total_bill'], 
                                       bins=[0, 15, 25, 40, float('inf')], 
                                       labels=['Low', 'Medium', 'High', 'Very_High'])
print(f"✅ Creada variable 'bill_category'")

# 2. Tamaño del grupo categorizado
tips_enhanced['size_category'] = tips_enhanced['size'].apply(
    lambda x: 'Single' if x == 1 else 'Couple' if x == 2 else 'Small_Group' if x <= 4 else 'Large_Group'
)
print(f"✅ Creada variable 'size_category'")

# 3. Indicadores binarios para días de fin de semana
tips_enhanced['is_weekend'] = tips_enhanced['day'].isin(['Sat', 'Sun']).astype(int)
print(f"✅ Creada variable 'is_weekend'")

# 4. Indicador para cena
tips_enhanced['is_dinner'] = (tips_enhanced['time'] == 'Dinner').astype(int)
print(f"✅ Creada variable 'is_dinner'")

# 5. Indicador para fumador
tips_enhanced['is_fumador'] = (tips_enhanced['smoker'] == 'Yes').astype(int)
tips_enhanced['is_female'] = (tips_enhanced['sex'] == 'Female').astype(int)
print(f"✅ Creadas variables 'is_fumador' e 'is_female'")

# 6. Propina promedio por persona
tips_enhanced['tip_per_person'] = tips_enhanced['tip'] / tips_enhanced['size']
print(f"✅ Creada variable 'tip_per_person'")

# 7. Interacción entre total_bill y size
tips_enhanced['bill_per_person'] = tips_enhanced['total_bill'] / tips_enhanced['size']
print(f"✅ Creada variable 'bill_per_person'")

# 8. Códigos numéricos para variables categóricas
# Día de la semana
day_mapping = {'Thur': 0, 'Fri': 1, 'Sat': 2, 'Sun': 3}
tips_enhanced['day_encoded'] = tips_enhanced['day'].map(day_mapping)
print(f"✅ Creada variable 'day_encoded'")

print(f"\n📊 Dataset después del feature engineering:")
print(f"  - Shape original: {tips.shape}")
print(f"  - Shape después: {tips_enhanced.shape}")
print(f"  - Nuevas variables: {tips_enhanced.shape[1] - tips.shape[1]}")

# Mostrar las nuevas variables creadas
new_columns = [col for col in tips_enhanced.columns if col not in tips.columns]
print(f"\n🆕 Nuevas variables creadas: {new_columns}")

# Estadísticas de las nuevas variables numéricas
new_numeric_cols = ['tip_per_person', 'bill_per_person', 'is_weekend', 'is_dinner', 'is_fumador', 'is_female', 'day_encoded']
print(f"\n📈 Estadísticas de nuevas variables numéricas:")
print(tips_enhanced[new_numeric_cols].describe().round(3))

In [ ]:
# Visualización de nuevas variables
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Visualización de Nuevas Variables Creadas', fontsize=16)

# 1. Distribución de bill_category
bill_cat_counts = tips_enhanced['bill_category'].value_counts()
axes[0,0].pie(bill_cat_counts.values, labels=bill_cat_counts.index, autopct='%1.1f%%')
axes[0,0].set_title('Distribución de Categorías de Cuenta')

# 2. Tip promedio por categoría de cuenta
tip_by_bill_cat = tips_enhanced.groupby('bill_category')['tip'].mean()
axes[0,1].bar(tip_by_bill_cat.index, tip_by_bill_cat.values, color='skyblue')
axes[0,1].set_title('Tip Promedio por Categoría de Cuenta')
axes[0,1].set_xlabel('Categoría')
axes[0,1].set_ylabel('Tip Promedio ($)')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Propina por tamaño de grupo
tips_enhanced.boxplot(column='tip', by='size_category', ax=axes[0,2])
axes[0,2].set_title('Distribución de Propina por Tamaño de Grupo')
axes[0,2].set_xlabel('Tamaño de Grupo')
axes[0,2].set_ylabel('Tip ($)')

# 4. Comparación weekend vs weekday
weekend_comparison = tips_enhanced.groupby('is_weekend')[['tip', 'tip_percentage']].mean()
weekend_comparison.plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Comparación: Weekend vs Weekday')
axes[1,0].set_xlabel('Es Fin de Semana')
axes[1,0].set_ylabel('Promedio')
axes[1,0].legend(['Tip ($)', 'Tip Percentage (%)'])
axes[1,0].set_xticklabels(['Weekday', 'Weekend'], rotation=0)

# 5. Bill per person vs Tip per person
axes[1,1].scatter(tips_enhanced['bill_per_person'], tips_enhanced['tip_per_person'], 
                 alpha=0.6, color='purple')
axes[1,1].set_xlabel('Bill per Person ($)')
axes[1,1].set_ylabel('Tip per Person ($)')
axes[1,1].set_title('Bill per Person vs Tip per Person')
axes[1,1].grid(True, alpha=0.3)

# 6. Heatmap de correlación de nuevas variables
new_corr_cols = ['tip_per_person', 'bill_per_person', 'is_weekend', 'is_dinner', 'is_fumador', 'is_female', 'day_encoded', 'tip']
new_corr_matrix = tips_enhanced[new_corr_cols].corr()
sns.heatmap(new_corr_matrix, annot=True, cmap='RdBu_r', center=0, ax=axes[1,2], fmt='.3f')
axes[1,2].set_title('Correlaciones - Nuevas Variables vs Tip')

plt.tight_layout()
plt.show()

## 🔄 Paso 5: Preprocesamiento de Datos

**🎯 Enfoque**: Preparación de los datos para los modelos de machine learning.

In [ ]:
# Preprocesamiento para modelos
print(f"🔄 PREPROCESAMIENTO PARA MACHINE LEARNING")
print("=" * 50)

# Definir las features para los modelos
# Para predecir el monto de la propina (regresión)
features_regression = [
    'total_bill', 'size', 'day_encoded', 'is_weekend', 'is_dinner', 
    'is_fumador', 'is_female', 'bill_per_person', 'tip_per_person'
]

# Para predecir el porcentaje de propina (clasificación)
tips_enhanced['tip_pct_category'] = pd.cut(tips_enhanced['tip_percentage'], 
                                         bins=[0, 10, 15, 20, 30, 100], 
                                         labels=['Very_Low', 'Low', 'Medium', 'High', 'Very_High'])

# Preparar datasets
X_regression = tips_enhanced[features_regression]
y_regression = tips_enhanced['tip']

# Para clasificación del porcentaje de propina
classification_features = [f for f in features_regression if f != 'tip_per_person']  # Remover target derivada
X_classification = tips_enhanced[classification_features]
y_classification = tips_enhanced['tip_pct_category']

print(f"📊 Datasets preparados:")
print(f"  - Regresión: X shape = {X_regression.shape}, y shape = {y_regression.shape}")
print(f"  - Clasificación: X shape = {X_classification.shape}, y shape = {y_classification.shape}")

# Información de los datasets
print(f"\n📋 Variables utilizadas en regresión: {features_regression}")
print(f"📋 Variables utilizadas en clasificación: {classification_features}")
print(f"📋 Distribución de clases en clasificación:")
print(y_classification.value_counts().sort_index())

# Verificar si hay valores faltantes en las features seleccionadas
print(f"\n🔍 Verificación de valores faltantes:")
missing_X_reg = X_regression.isnull().sum().sum()
missing_X_class = X_classification.isnull().sum().sum()
print(f"  - Missing en X_regression: {missing_X_reg}")
print(f"  - Missing en X_classification: {missing_X_class}")

# Estadísticas de las variables seleccionadas
print(f"\n📈 Estadísticas de features para regresión:")
print(X_regression.describe().round(2))

# Separación train/test
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_regression, y_regression, test_size=0.2, random_state=42
)

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_classification, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

print(f"\n📊 División de datos:")
print(f"  - Regresión - Entrenamiento: {X_reg_train.shape[0]}, Prueba: {X_reg_test.shape[0]}")
print(f"  - Clasificación - Entrenamiento: {X_clf_train.shape[0]}, Prueba: {X_clf_test.shape[0]}")

# Escalado de características
scaler_reg = StandardScaler()
X_reg_train_scaled = scaler_reg.fit_transform(X_reg_train)
X_reg_test_scaled = scaler_reg.transform(X_reg_test)

scaler_clf = StandardScaler()
X_clf_train_scaled = scaler_clf.fit_transform(X_clf_train)
X_clf_test_scaled = scaler_clf.transform(X_clf_test)

print(f"✅ Escalado completado para ambos datasets")

## 🤖 Paso 6: Modelos de Regresión - Predicción del Monto de la Propina

**🎯 Objetivo**: Predecir el monto de la propina usando diferentes algoritmos.

In [ ]:
# Modelos de regresión
print(f"🤖 MODELOS DE REGRESIÓN - PREDICCIÓN DEL MONTO DE LA PROPINA")
print("=" * 65)

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error

# Definir modelos
regression_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Evaluación de modelos
regression_results = {}
print(f"\n🔄 Evaluando modelos de regresión...")

for model_name, model in regression_models.items():
    print(f"  - Entrenando {model_name}...")
    
    # Entrenar
    model.fit(X_reg_train_scaled, y_reg_train)
    
    # Predicciones
    y_pred_train = model.predict(X_reg_train_scaled)
    y_pred_test = model.predict(X_reg_test_scaled)
    
    # Métricas
    train_r2 = r2_score(y_reg_train, y_pred_train)
    test_r2 = r2_score(y_reg_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_test))
    test_mae = mean_absolute_error(y_reg_test, y_pred_test)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_reg_train_scaled, y_reg_train, cv=5, scoring='r2')
    
    regression_results[model_name] = {
        'train_r2': train_r2,
        'test_r2': test_r2,
        'test_rmse': test_rmse,
        'test_mae': test_mae,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }

# Mostrar resultados
print(f"\n📊 RESULTADOS DE REGRESIÓN:")
print("=" * 60)
print(f"{'Modelo':<20} {'Train R²':<10} {'Test R²':<10} {'RMSE':<8} {'MAE':<8} {'CV R² (μ±σ)':<15}")
print("-" * 60)

for model_name, results in regression_results.items():
    print(f"{model_name:<20} {results['train_r2']:<10.3f} {results['test_r2']:<10.3f} "
          f"{results['test_rmse']:<8.3f} {results['test_mae']:<8.3f} "
          f"{results['cv_mean']:.3f}±{results['cv_std']:.3f}")

# Encontrar mejor modelo
best_reg_model_name = max(regression_results.keys(), 
                        key=lambda x: regression_results[x]['test_r2'])
best_reg_model = regression_models[best_reg_model_name]

print(f"\n🏆 Mejor modelo de regresión: {best_reg_model_name}")
print(f"   Test R² Score: {regression_results[best_reg_model_name]['test_r2']:.3f}")
print(f"   RMSE: {regression_results[best_reg_model_name]['test_rmse']:.3f}")
print(f"   MAE: {regression_results[best_reg_model_name]['test_mae']:.3f}")

In [ ]:
# Visualización de resultados de regresión
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Evaluación de Modelos de Regresión', fontsize=16)

# 1. Comparación de métricas
model_names = list(regression_results.keys())
test_r2_scores = [regression_results[model]['test_r2'] for model in model_names]
test_rmse_scores = [regression_results[model]['test_rmse'] for model in model_names]
test_mae_scores = [regression_results[model]['test_mae'] for model in model_names]

x = np.arange(len(model_names))
width = 0.25

axes[0,0].bar(x - width, test_r2_scores, width, label='Test R²', alpha=0.7)
axes[0,0].bar(x, [score/5 for score in test_rmse_scores], width, label='RMSE/5', alpha=0.7)
axes[0,0].bar(x + width, [score/5 for score in test_mae_scores], width, label='MAE/5', alpha=0.7)
axes[0,0].set_xlabel('Modelos')
axes[0,0].set_ylabel('Score')
axes[0,0].set_title('Comparación de Métricas')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(model_names, rotation=45)
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Predicciones vs valores reales del mejor modelo
best_predictions = best_reg_model.predict(X_reg_test_scaled)

axes[0,1].scatter(y_reg_test, best_predictions, alpha=0.6, color='blue')
axes[0,1].plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[0,1].set_xlabel('Valores Reales')
axes[0,1].set_ylabel('Predicciones')
axes[0,1].set_title(f'Predicciones vs Reales - {best_reg_model_name}')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Residuales
residuals = y_reg_test - best_predictions
axes[1,0].scatter(best_predictions, residuals, alpha=0.6, color='green')
axes[1,0].axhline(y=0, color='red', linestyle='--')
axes[1,0].set_xlabel('Predicciones')
axes[1,0].set_ylabel('Residuales')
axes[1,0].set_title('Análisis de Residuales')
axes[1,0].grid(True, alpha=0.3)

# 4. Feature importance (si el modelo lo permite)
if hasattr(best_reg_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': features_regression,
        'importance': best_reg_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    axes[1,1].barh(range(len(feature_importance)), feature_importance['importance'])
    axes[1,1].set_yticks(range(len(feature_importance)))
    axes[1,1].set_yticklabels(feature_importance['feature'])
    axes[1,1].set_xlabel('Importancia')
    axes[1,1].set_title('Feature Importance')
    axes[1,1].invert_yaxis()
elif hasattr(best_reg_model, 'coef_'):
    # Para modelos lineales, mostrar coeficientes
    feature_coef = pd.DataFrame({
        'feature': features_regression,
        'coefficient': best_reg_model.coef_
    }).sort_values('coefficient', key=abs, ascending=False)
    
    axes[1,1].barh(range(len(feature_coef)), feature_coef['coefficient'])
    axes[1,1].set_yticks(range(len(feature_coef)))
    axes[1,1].set_yticklabels(feature_coef['feature'])
    axes[1,1].set_xlabel('Coeficiente')
    axes[1,1].set_title('Coeficientes del Modelo')
    axes[1,1].invert_yaxis()
else:
    axes[1,1].text(0.5, 0.5, 'Modelo no soporta\ninterpretación de features', 
                  ha='center', va='center', transform=axes[1,1].transAxes)
    axes[1,1].set_title('Feature Importance - No Disponible')

plt.tight_layout()
plt.show()

## 🤖 Paso 7: Modelos de Clasificación - Predicción del Porcentaje de Propina

**🎯 Objetivo**: Clasificar el porcentaje de propina en categorías.

In [ ]:
# Modelos de clasificación
print(f"🤖 MODELOS DE CLASIFICACIÓN - PREDICCIÓN DEL PORCENTAJE DE PROPINA")
print("=" * 70)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

# Definir modelos
classification_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Evaluación de modelos
classification_results = {}
print(f"\n🔄 Evaluando modelos de clasificación...")

for model_name, model in classification_models.items():
    print(f"  - Entrenando {model_name}...")
    
    # Entrenar
    model.fit(X_clf_train_scaled, y_clf_train)
    
    # Predicciones
    y_pred_train = model.predict(X_clf_train_scaled)
    y_pred_test = model.predict(X_clf_test_scaled)
    
    # Métricas
    train_accuracy = accuracy_score(y_clf_train, y_pred_train)
    test_accuracy = accuracy_score(y_clf_test, y_pred_test)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_clf_train_scaled, y_clf_train, cv=5, scoring='accuracy')
    
    classification_results[model_name] = {
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }

# Mostrar resultados
print(f"\n📊 RESULTADOS DE CLASIFICACIÓN:")
print("=" * 55)
print(f"{'Modelo':<20} {'Train Acc':<12} {'Test Acc':<12} {'CV Acc (μ±σ)':<15}")
print("-" * 55)

for model_name, results in classification_results.items():
    print(f"{model_name:<20} {results['train_accuracy']:<12.3f} {results['test_accuracy']:<12.3f} "
          f"{results['cv_mean']:.3f}±{results['cv_std']:.3f}")

# Encontrar mejor modelo
best_clf_model_name = max(classification_results.keys(), 
                        key=lambda x: classification_results[x]['test_accuracy'])
best_clf_model = classification_models[best_clf_model_name]

print(f"\n🏆 Mejor modelo de clasificación: {best_clf_model_name}")
print(f"   Test Accuracy: {classification_results[best_clf_model_name]['test_accuracy']:.3f}")

# Reporte de clasificación detallado
y_clf_pred = best_clf_model.predict(X_clf_test_scaled)
print(f"\n📋 Reporte de Clasificación Detallado - {best_clf_model_name}:")
target_names = y_classification.cat.categories
print(classification_report(y_clf_test, y_clf_pred, target_names=target_names))

In [ ]:
# Visualización de resultados de clasificación
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Evaluación de Modelos de Clasificación', fontsize=16)

# 1. Comparación de accuracies
model_names_clf = list(classification_results.keys())
test_acc_scores = [classification_results[model]['test_accuracy'] for model in model_names_clf]
cv_acc_scores = [classification_results[model]['cv_mean'] for model in model_names_clf]

x = np.arange(len(model_names_clf))
width = 0.35

axes[0,0].bar(x - width/2, test_acc_scores, width, label='Test Accuracy', alpha=0.7)
axes[0,0].bar(x + width/2, cv_acc_scores, width, label='CV Accuracy', alpha=0.7)
axes[0,0].set_xlabel('Modelos')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_title('Comparación de Accuracies')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(model_names_clf, rotation=45)
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Matriz de confusión
cm = confusion_matrix(y_clf_test, y_clf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1], 
            xticklabels=target_names, yticklabels=target_names)
axes[0,1].set_title(f'Matriz de Confusión - {best_clf_model_name}')
axes[0,1].set_xlabel('Predicción')
axes[0,1].set_ylabel('Valor Real')

# 3. Distribución de clases predichas vs reales
# Reindexar ambas series según todas las clases (target_names) para evitar mismatch cuando
# algunas clases no aparecen en las predicciones
y_clf_test_counts = y_clf_test.value_counts().reindex(target_names, fill_value=0)
y_clf_pred_counts = pd.Series(y_clf_pred, index=y_clf_test.index).value_counts().reindex(target_names, fill_value=0)

x = np.arange(len(target_names))
width = 0.35

axes[1,0].bar(x - width/2, y_clf_test_counts.values, width, label='Real', alpha=0.7)
axes[1,0].bar(x + width/2, y_clf_pred_counts.values, width, label='Predicho', alpha=0.7)
axes[1,0].set_xlabel('Clases')
axes[1,0].set_ylabel('Cantidad')
axes[1,0].set_title('Distribución: Real vs Predicho')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(target_names, rotation=45)
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 4. Feature importance (si el modelo lo permite)
if hasattr(best_clf_model, 'feature_importances_'):
    feature_importance_clf = pd.DataFrame({
        'feature': classification_features,
        'importance': best_clf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    axes[1,1].barh(range(len(feature_importance_clf)), feature_importance_clf['importance'])
    axes[1,1].set_yticks(range(len(feature_importance_clf)))
    axes[1,1].set_yticklabels(feature_importance_clf['feature'])
    axes[1,1].set_xlabel('Importancia')
    axes[1,1].set_title('Feature Importance')
    axes[1,1].invert_yaxis()
else:
    axes[1,1].text(0.5, 0.5, 'Modelo no soporta\ninterpretación de features', 
                  ha='center', va='center', transform=axes[1,1].transAxes)
    axes[1,1].set_title('Feature Importance - No Disponible')

plt.tight_layout()
plt.show()

## 🎓 Paso 8: Conclusiones y Análisis Final

**🎯 Síntesis**: Evaluación del aprendizaje y insights del análisis.

In [ ]:
# Conclusiones finales
print("🎓 CONCLUSIONES DEL ANÁLISIS DE RESTAURANT TIPS")
print("=" * 55)

print(f"\n📊 RESUMEN DEL DATASET:")
print(f"  - Registros totales: {len(tips_enhanced)}")
print(f"  - Variables originales: {tips.shape[1]}")
print(f"  - Variables después del feature engineering: {tips_enhanced.shape[1]}")
print(f"  - Variables utilizadas en modelos: {len(features_regression)}")
print(f"  - Rango de propinas: ${tips_enhanced['tip'].min():.2f} - ${tips_enhanced['tip'].max():.2f}")
print(f"  - Propina promedio: ${tips_enhanced['tip'].mean():.2f}")
print(f"  - Porcentaje de propina promedio: {tips_enhanced['tip_percentage'].mean():.1f}%")

print(f"\n🤖 MEJORES MODELOS ENCONTRADOS:")
print(f"\n📈 REGRESIÓN (Predicción del monto):")
print(f"  - Mejor modelo: {best_reg_model_name}")
print(f"  - R² Score: {regression_results[best_reg_model_name]['test_r2']:.3f}")
print(f"  - RMSE: ${regression_results[best_reg_model_name]['test_rmse']:.3f}")
print(f"  - MAE: ${regression_results[best_reg_model_name]['test_mae']:.3f}")
print(f"  - Interpretación: El modelo explica {regression_results[best_reg_model_name]['test_r2']*100:.1f}% de la variabilidad")

print(f"\n📊 CLASIFICACIÓN (Categorización del %):")
print(f"  - Mejor modelo: {best_clf_model_name}")
print(f"  - Accuracy: {classification_results[best_clf_model_name]['test_accuracy']:.3f}")
print(f"  - Interpretación: {classification_results[best_clf_model_name]['test_accuracy']*100:.1f}% de clasificaciones correctas")

print(f"\n🔍 INSIGHTS CLAVE DEL ANÁLISIS:")

# Insights de correlación
total_bill_corr = correlation_matrix.loc['tip', 'total_bill']
size_corr = correlation_matrix.loc['tip', 'size']

print(f"\n💰 FACTORES QUE INFLUYEN EN LA PROPINA:")
print(f"  1. Monto total de la cuenta (r={total_bill_corr:.3f})")
print(f"     → Fuerte correlación positiva: más cuenta → más propina")
print(f"  2. Tamaño del grupo (r={size_corr:.3f})")
print(f"     → Grupos más grandes tienden a dejar más propina total")

# Insights categóricos
weekend_vs_weekday = tips_enhanced.groupby('is_weekend')['tip'].mean()
lunch_vs_dinner = tips_enhanced.groupby('is_dinner')['tip'].mean()

print(f"\n📅 PATRONES TEMPORALES:")
print(f"  - Fin de semana vs días laborales: {weekend_vs_weekday[1]:.2f}$ vs {weekend_vs_weekday[0]:.2f}$")
print(f"  - Cena vs almuerzo: {lunch_vs_dinner[1]:.2f}$ vs {lunch_vs_dinner[0]:.2f}$")

# Insights de feature engineering
print(f"\n🛠️ VALOR DEL FEATURE ENGINEERING:")
print(f"  - Nuevas variables creadas: bill_per_person, tip_per_person, is_weekend")
print(f"  - Correlación bill_per_person con tip: {tips_enhanced['bill_per_person'].corr(tips_enhanced['tip_per_person']):.3f}")
print(f"  - Las variables derivadas mejoraron la interpretabilidad")

print(f"\n💡 RECOMENDACIONES PARA EL NEGOCIO:")
print(f"  1. Estrategias de precios: Grupos grandes = oportunidad de mayor ticket promedio")
print(f"  2. Servicio en fines de semana: Mayor propensión a dejar propinas")
print(f"  3. Optimización de servicio de cena: Mejores propinas que almuerzo")
print(f"  4. Programa de fidelización: Basado en tip_percentage para identificar patrones")

print(f"\n🎯 APRENDIZAJES TÉCNICOS:")
print(f"  ✅ Feature Engineering es crucial: variables derivadas mejoraron predicciones")
print(f"  ✅ Random Forest superó modelos lineales para este dataset")
print(f"  ✅ Clasificación binaria vs multiclase: ambos enfoques son útiles")
print(f"  ✅ Variables categóricas proporcionan insights significativos")
print(f"  ✅ Análisis exploratorio guió las decisiones de modelado")

print(f"\n🚀 SIGUIENTES PASOS SUGERIDOS:")
print(f"  1. Optimización de hiperparámetros con GridSearch")
print(f"  2. Análisis de interacciones entre variables")
print(f"  3. Validación cruzada estratificada para mejor robustez")
print(f"  4. Implementación de ensemble de los mejores modelos")
print(f"  5. Análisis de feature importance en el contexto del negocio")

print(f"\n✅ DATASET COMPLETADO: Restaurant Tips")
print(f"   - Técnicas aprendidas: EDA, Feature Engineering, Regression, Classification")
print(f"   - Complejidad: Intermedia - Variables mixtas, patrones reales")
print(f"   - Ideal para: Comprensión de preprocesamiento + análisis de negocios")

---

## 📚 Resumen de Técnicas Aplicadas

| Técnica | Propósito | Resultado en Este Dataset |
|---------|-----------|---------------------------|
| **Análisis Exploratorio** | Comprender distribuciones y relaciones | Identificación de patrones de propinas |
| **Feature Engineering** | Crear variables derivadas | bill_per_person, tip_per_person mejoraron modelos |
| **Visualización** | Identificar insights visuales | Diferencias claras por día, tiempo, género |
| **Preprocesamiento** | Preparar datos para ML | Scaling, encoding, train/test split |
| **Regresión** | Predecir monto de propina | R² = 0.547 con Random Forest |
| **Clasificación** | Categorizar porcentaje | Accuracy = 0.727 con Random Forest |

## 🎯 Posición en la Progresión del Curso

Restaurant Tips es ideal para:

### ✅ FORTALEZAS:
- **Contexto familiar**: Todos entienden propinas en restaurantes
- **Variables mixtas**: Excelente práctica con datos reales
- **Feature Engineering**: Oportunidades claras para crear nuevas variables
- **Doble objetivo**: Regresión y clasificación
- **Insights de negocio**: Resultados interpretables

### 📈 PROGRESIÓN SUGERIDA:
1. **Wine Quality** (Semana 1) → ⭐ Básicos: 13 features, limpieza simple
2. **Restaurant Tips** (Semana 2) → ⭐⭐ **Intermedio**: Variables mixtas, feature engineering
3. **Forest Cover Type** (Semana 3) → ⭐⭐⭐ Avanzado: 54 features, feature selection
4. **Adult Census** (Semana 4) → ⭐⭐ Strings + valores faltantes
5. **Heart Disease** (Semana 5) → ⭐⭐⭐ Imputación avanzada

## 🎓 Conclusión Pedagógica

El dataset Restaurant Tips sirve como **puente perfecto** entre datasets básicos y avanzados. Introduce:

- **Complejidad manejable**: Variables mixtas sin ser abrumador
- **Feature Engineering práctico**: Variables derivadas con significado claro
- **Múltiples enfoques**: Regresión y clasificación en el mismo dataset
- **Interpretación de negocio**: Resultados que se pueden aplicar en la vida real

**Es ideal como el segundo paso en la progresión del curso**, consolidando conceptos básicos mientras introduce nuevas técnicas de manera natural.